# 08｜降維與資料表示

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/08_降維與資料表示.md`，目前 45 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

邊界體積、平面投影、SVD/IPCA、LLE 權重、多法降維與 JL 平方距離。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 08-03 | 4、5、6 | 1. 維度災難：大部分體積靠近邊界 |
| 08-04 | 13、14、16、17、18、19、20 | 2. 手算驗算：PCA 的方向與共變異矩陣 |
| 08-05 | 22、30 | 3. 手算驗算：解釋變異比例選維度 |
| 08-06 | 3、23、24、25、26 | 4. PCA on digits：先切分，再學投影，用累積曲線選 d |
| 08-07 | 29、30、43、44 | 5. 壓縮率與辨識品質：一起變好嗎？ |
| 08-08 | 9、10、35、43 | 6. Swiss roll：PCA 攤不開彎曲流形，LLE 保留局部鄰近並比較鄰居數 |
| 08-09 | 32、33、34 | 7. 隨機投影與 Johnson–Lindenstrauss |
| 08-10 | 4、6、7、8、9、10、11、12、13、45 | 8. 看見邊界、平面投影與流形標籤 |
| 08-11 | 16、18、20、21、27、28、32、33、34、45 | 9. SVD 驗算與分批投影 |
| 08-12 | 35、36、37、38、39、40、41、42、43、45 | 10. LLE 權重、監督式 LDA、t-SNE 與 UMAP |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 08-01｜投影片 08：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2', 'umap': 'umap-learn>=0.5.12,<0.6'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 08-02｜投影片 08：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 1. 維度災難：大部分體積靠近邊界

在 $[0,1]^n$ 均勻取點，隨維度上升，點更常靠近某個面，且點對之間的平均距離變大、相對變異（標準差／平均）變小。

In [ ]:
# 程式 08-03｜投影片 08：4、5、6 頁
rng = np.random.default_rng(SEED)
rows = []
for n in [2, 10, 100, 1000]:
    pts = rng.uniform(0, 1, (2000, n))
    near_boundary = np.mean(np.any((pts < 0.05) | (pts > 0.95), axis=1))
    sample = pts[:400]
    from sklearn.metrics import pairwise_distances
    d = pairwise_distances(sample)  # 不建立 m×m×n 的巨大中間陣列
    off = d[np.triu_indices(len(sample), 1)]
    rows.append({'n_dims': n, 'frac_point_near_a_face': near_boundary,
                 'mean_pairwise_distance': off.mean(), 'distance_cv': off.std() / off.mean()})
curse = pd.DataFrame(rows).set_index('n_dims')
assert curse['frac_point_near_a_face'].is_monotonic_increasing
assert curse['distance_cv'].iloc[-1] < curse['distance_cv'].iloc[0]   # 相對差異變小
curse

## 2. 手算驗算：PCA 的方向與共變異矩陣

對應投影片活動「先猜方向」與「同一條線的共變異矩陣」。四個已中心化的點 $(-2,-2),(-1,-1),(1,1),(2,2)$。

In [ ]:
# 程式 08-04｜投影片 08：13、14、16、17、18、19、20 頁
Xc = np.array([[-2., -2.], [-1., -1.], [1., 1.], [2., 2.]])
C = Xc.T @ Xc / (len(Xc) - 1)                       # 樣本共變異，分母 m-1 = 3
eigval, eigvec = np.linalg.eigh(C)
order = np.argsort(eigval)[::-1]
eigval, eigvec = eigval[order], eigvec[:, order]
first_pc = eigvec[:, 0]
assert np.allclose(C, (10 / 3) * np.ones((2, 2)))
assert np.allclose(sorted(eigval), [0, 20 / 3])
assert np.allclose(np.abs(first_pc), [1 / np.sqrt(2)] * 2)          # 方向 (1,1)/sqrt(2)，符號不定
# 投影到對角線 vs 投影到 x 軸的重建誤差
proj_diag = (Xc @ first_pc)[:, None] * first_pc
proj_xaxis = Xc * np.array([1, 0])
assert np.allclose(proj_diag, Xc) and not np.allclose(proj_xaxis, Xc)
pd.Series({'cov_matrix': C.tolist(), 'eigenvalues': eigval.tolist(),
           'first_PC': first_pc.tolist(),
           'reconstruction_MSE_diagonal': float(np.mean((proj_diag - Xc) ** 2)),
           'reconstruction_MSE_xaxis': float(np.mean((proj_xaxis - Xc) ** 2))})

## 3. 手算驗算：解釋變異比例選維度

對應投影片離線題。特徵值 `[8, 1.5, 0.5]`，保留 90% 至少需要幾維？

In [ ]:
# 程式 08-05｜投影片 08：22、30 頁
eig = np.array([8., 1.5, 0.5])
cum = np.cumsum(eig) / eig.sum()
need_90 = int(np.searchsorted(cum, 0.90) + 1)
assert np.allclose(cum, [0.8, 0.95, 1.0]) and need_90 == 2
pd.Series({'cumulative_ratio': cum.tolist(), 'dims_for_90pct': need_90, 'dims_for_80pct': 1})

## 4. PCA on digits：先切分，再學投影，用累積曲線選 d

`PCA(n_components=0.95)` 依訓練資料的累積解釋變異挑維度；`inverse_transform` 重建時已把平均值加回。

In [ ]:
# 程式 08-06｜投影片 08：3、23、24、25、26 頁
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(digits.data, digits.target, test_size=.3, stratify=digits.target, random_state=SEED)

full = PCA(svd_solver='full').fit(Xtr)
cumvar = np.cumsum(full.explained_variance_ratio_)
d95 = int(np.argmax(cumvar >= 0.95) + 1)
plt.plot(np.arange(1, len(cumvar) + 1), cumvar)
plt.axhline(.95, ls=':', c='grey'); plt.axvline(d95, ls='--', c='k', label=f'd={d95} keeps 95%')
plt.xlabel('number of components'); plt.ylabel('cumulative explained variance'); plt.legend()
savefig('13_pca_scree')

pca = PCA(n_components=0.95, svd_solver='full').fit(Xtr)
Zte = pca.transform(Xte)
rebuilt = pca.inverse_transform(Zte)
fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
for k in range(6):
    axes[0, k].imshow(Xte[k].reshape(8, 8), cmap='binary'); axes[0, k].axis('off')
    axes[1, k].imshow(rebuilt[k].reshape(8, 8), cmap='binary'); axes[1, k].axis('off')
axes[0, 0].set_ylabel('original'); axes[1, 0].set_ylabel('rebuilt')
savefig('13_pca_reconstruction')
assert pca.n_components_ == d95
print('components for 95% variance:', d95, ' of', Xtr.shape[1])

## 5. 壓縮率與辨識品質：一起變好嗎？

對已切分的 digits，比較 2／10／30 主成分：解釋變異、重建 MSE、下游分類 CV 與時間。PCA 放在 CV 內每折重 fit。

In [ ]:
# 程式 08-07｜投影片 08：29、30、43、44 頁
import time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
rows = []
for d in [2, 10, 30]:
    p = PCA(n_components=d, random_state=SEED).fit(Xtr)
    rebuilt = p.inverse_transform(p.transform(Xtr))
    clf = make_pipeline(PCA(n_components=d, random_state=SEED), StandardScaler(), LogisticRegression(max_iter=2000))
    t0 = time.perf_counter(); cv = cross_val_score(clf, Xtr, ytr, cv=5); sec = time.perf_counter() - t0
    rows.append({'components': d, 'explained_var': float(p.explained_variance_ratio_.sum()),
                 'reconstruction_MSE': float(np.mean((rebuilt - Xtr) ** 2)),
                 'classification_CV_acc': cv.mean(), 'cv_seconds': round(sec, 3)})
comp = pd.DataFrame(rows).set_index('components')
assert comp['reconstruction_MSE'].is_monotonic_decreasing
# 分類 CV 不保證隨維度單調增加；這是要觀察的結果。
comp

## 6. Swiss roll：PCA 攤不開彎曲流形，LLE 保留局部鄰近

PCA 找變異最大的平面投影；彎曲流形需要保留局部鄰居關係的方法，例如 LLE。

In [ ]:
# 程式 08-08｜投影片 08：9、10、35、43 頁
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import LocallyLinearEmbedding
Xsr, color = make_swiss_roll(n_samples=1200, noise=0.05, random_state=SEED)
pca2 = PCA(n_components=2, random_state=SEED).fit_transform(Xsr)
lle2 = LocallyLinearEmbedding(n_neighbors=12, n_components=2, random_state=SEED).fit_transform(Xsr)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].scatter(Xsr[:, 0], Xsr[:, 2], c=color, cmap='Spectral', s=8); axes[0].set_title('Swiss roll (x, z)')
axes[1].scatter(pca2[:, 0], pca2[:, 1], c=color, cmap='Spectral', s=8); axes[1].set_title('PCA to 2D')
axes[2].scatter(lle2[:, 0], lle2[:, 1], c=color, cmap='Spectral', s=8); axes[2].set_title('LLE to 2D')
savefig('13_swiss_roll')

def neighbor_agreement(Z, k=10):
    from sklearn.neighbors import NearestNeighbors
    nn_hi = NearestNeighbors(n_neighbors=k + 1).fit(Xsr).kneighbors(Xsr, return_distance=False)[:, 1:]
    nn_lo = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z, return_distance=False)[:, 1:]
    return float(np.mean([len(set(a) & set(b)) / k for a, b in zip(nn_hi, nn_lo)]))
agree = {'PCA': neighbor_agreement(pca2), 'LLE': neighbor_agreement(lle2)}
assert agree['LLE'] > agree['PCA']            # LLE 較好保留局部鄰居
print('fraction of 10 nearest neighbors preserved:', {k: round(v, 3) for k, v in agree.items()})
lle_rows8 = []
for neighbors8 in [5, 10, 20, 30]:
    model8 = LocallyLinearEmbedding(n_neighbors=neighbors8, n_components=2, random_state=SEED)
    z8 = model8.fit_transform(Xsr)
    lle_rows8.append({'n_neighbors': neighbors8, 'reconstruction_error': model8.reconstruction_error_,
                      'neighbor_agreement': neighbor_agreement(z8)})
print(pd.DataFrame(lle_rows8))

## 7. 隨機投影與 Johnson–Lindenstrauss

不用先找主方向，隨機矩陣也能把高維點投到較低維而大致保距。JL 引理給出「保距的一個足夠維度界線（不是必要的最小維度）」，只與樣本數與容忍度有關，不看原始維度。

In [ ]:
# 程式 08-09｜投影片 08：32、33、34 頁
from sklearn.random_projection import SparseRandomProjection, johnson_lindenstrauss_min_dim
rng = np.random.default_rng(SEED)
Xhi = rng.normal(size=(600, 4000))
eps = 0.2
k_min = int(johnson_lindenstrauss_min_dim(n_samples=600, eps=eps))
rp = SparseRandomProjection(n_components=k_min, random_state=SEED)
Xlo = rp.fit_transform(Xhi)
i = rng.integers(0, len(Xhi), 2000)
j = (i + rng.integers(1, len(Xhi), 2000)) % len(Xhi)
d_hi = np.linalg.norm(Xhi[i] - Xhi[j], axis=1)
d_lo = np.linalg.norm(Xlo[i] - Xlo[j], axis=1)
within = float(np.mean(np.abs((d_lo / d_hi) ** 2 - 1) <= eps))
report('dimensionality', {'curse': curse.to_dict(), 'first_PC': first_pc.tolist(),
       'digits_d95': d95, 'component_tradeoff': comp.to_dict(),
       'manifold_neighbor_agreement': agree, 'JL_min_dim': k_min, 'squared_distance_pairs_within_eps': within})
assert within > 0.9
print('JL min dim for n=600, eps=0.2:', k_min, ' | pairs within eps:', round(within, 3))

## 8. 看見邊界、平面投影與流形標籤

體積界線可精確計算；3D 點投影到 PCA 平面會丟掉法向分量。Swiss roll 的顏色標籤由內在座標決定，展開前後同一條界線的形狀不同。以下皆為合成資料的概念實驗。

In [ ]:
# 程式 08-10｜投影片 08：4、6、7、8、9、10、11、12、13、45 頁
dims = np.arange(1, 101)
plt.plot(dims, 1 - .9 ** dims, label='1 - 0.9^d')
plt.scatter(curse.index[:3], curse.frac_point_near_a_face.iloc[:3], label='Monte Carlo')
plt.xlabel('dimensions'); plt.ylabel('probability near a face'); plt.legend()
savefig('c08_boundary_volume')
rng8 = np.random.default_rng(SEED)
plane = rng8.normal(size=(160, 2)) @ np.array([[2, 0, 1], [0, .6, .8]])
plane += rng8.normal(scale=.08, size=plane.shape)
p8 = PCA(2).fit(plane); z8 = p8.transform(plane); back8 = p8.inverse_transform(z8)
fig = plt.figure(figsize=(11, 4)); ax = fig.add_subplot(121, projection='3d')
ax.scatter(*plane.T, c=z8[:, 0], s=9); ax.scatter(*back8.T, s=5, color='gray')
for a8, b8 in zip(plane[::6], back8[::6]): ax.plot(*np.stack([a8, b8]).T, color='red')
ax.set_title('3D points and plane projection')
ax = fig.add_subplot(122); ax.scatter(*z8.T, c=z8[:, 0], s=9); ax.set(xlabel='PC1', ylabel='PC2', title='Coordinates on the plane')
savefig('c08_pca_projection')
# 最大變異方向的座標；斜方向的投影誤差隨角度變動
angles = np.linspace(0, np.pi, 181); directions = np.c_[np.cos(angles), np.sin(angles)]
var8 = np.var(Xc @ directions.T, axis=0, ddof=1)
plt.plot(np.degrees(angles), var8); plt.xlabel('projection angle (degrees)'); plt.ylabel('sample variance')
savefig('c08_projection_variance')
fig = plt.figure(figsize=(11, 4)); ax = fig.add_subplot(121, projection='3d')
labels8 = color > np.median(color)
ax.scatter(*Xsr.T, c=labels8, s=7, cmap='coolwarm'); ax.set_title('Same labels on Swiss roll')
ax = fig.add_subplot(122); ax.scatter(color, Xsr[:, 1], c=labels8, s=7, cmap='coolwarm')
ax.axvline(np.median(color), c='k'); ax.set(xlabel='intrinsic roll coordinate', ylabel='height', title='Unrolled class boundary')
savefig('c08_manifold_boundary')

## 9. SVD 驗算與分批投影

SVD 向量可以整對換號；比較重建與投影矩陣，比直接比較向量可靠。IncrementalPCA 先完成所有 partial_fit，再轉換測試集。

In [ ]:
# 程式 08-11｜投影片 08：16、18、20、21、27、28、32、33、34、45 頁
A8 = np.array([[-1., 1.], [0., -1.], [1., 1.]])
U8, s8, Vt8 = np.linalg.svd(A8, full_matrices=False)
assert np.allclose(U8 @ np.diag(s8) @ Vt8, A8)
assert np.allclose(s8 ** 2, [3, 2])
Uc, sc, Vtc = np.linalg.svd(Xc, full_matrices=False)
assert np.allclose(sc ** 2 / (len(Xc) - 1), eigval)
from sklearn.decomposition import IncrementalPCA
ip8 = IncrementalPCA(n_components=10)
for batch in np.array_split(Xtr, 6): ip8.partial_fit(batch)
methods8 = {'full': PCA(10, svd_solver='full').fit(Xtr),
            'randomized': PCA(10, svd_solver='randomized', random_state=SEED).fit(Xtr), 'incremental': ip8}
print(pd.DataFrame([{'method': name, 'test_reconstruction_MSE': np.mean((Xte - p.inverse_transform(p.transform(Xte))) ** 2)} for name, p in methods8.items()]))
plt.hist((d_lo / d_hi) ** 2, bins=30); plt.axvline(1-eps, c='r'); plt.axvline(1+eps, c='r')
plt.xlabel('projected / original squared distance'); plt.ylabel('sampled pairs')
savefig('c08_jl_distortion')

## 10. LLE 權重、監督式 LDA、t-SNE 與 UMAP

直接驗算 LLE 的局部權重和為 1，再組出全域矩陣。t-SNE 與 UMAP 的全圖只作探索；LDA 需要標籤，不能稱為無監督方法。不同方法的圖不表示距離可直接比較。

In [ ]:
# 程式 08-12｜投影片 08：35、36、37、38、39、40、41、42、43、45 頁
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE, Isomap, MDS
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from umap import UMAP
small8 = Xsr[:180]; k8 = 10
neighbors8 = NearestNeighbors(n_neighbors=k8+1).fit(small8).kneighbors(small8, return_distance=False)[:, 1:]
W8 = np.zeros((len(small8), len(small8)))
for i, js in enumerate(neighbors8):
    offsets = small8[js] - small8[i]; gram = offsets @ offsets.T
    gram += np.eye(k8) * max(np.trace(gram) * 1e-3, 1e-8)
    weights = np.linalg.solve(gram, np.ones(k8)); weights /= weights.sum(); W8[i, js] = weights
assert np.allclose(W8.sum(axis=1), 1)
B8 = np.eye(len(W8)) - W8; eval8, evec8 = np.linalg.eigh(B8.T @ B8)
Zlle8 = evec8[:, 1:3]  # 去掉常數方向；縮放與旋轉不唯一
print('LLE objective:', np.sum((Zlle8 - W8 @ Zlle8)**2), 'first row weights:', W8[0, neighbors8[0]])
subset8, _, labels_subset8, _ = train_test_split(Xtr, ytr, train_size=400, stratify=ytr, random_state=SEED)
embeddings8 = {'PCA': PCA(2).fit_transform(subset8),
 'LLE': LocallyLinearEmbedding(n_neighbors=20, n_components=2, random_state=SEED).fit_transform(subset8),
 'Isomap': Isomap(n_neighbors=12).fit_transform(subset8),
 'MDS': MDS(n_components=2, n_init=1, max_iter=120, random_state=SEED).fit_transform(subset8),
 'LDA (uses labels)': LinearDiscriminantAnalysis(n_components=2).fit_transform(subset8, labels_subset8),
 't-SNE p=30': TSNE(perplexity=30, random_state=SEED).fit_transform(subset8),
 'UMAP n=15': UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=SEED).fit_transform(subset8)}
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (name, z8) in zip(axes.flat, embeddings8.items()):
    ax.scatter(*z8.T, c=labels_subset8, cmap='tab10', s=8); ax.set_title(name)
for ax in axes.flat[len(embeddings8):]: ax.axis('off')
savefig('c08_embedding_comparison')
# Gaussian 鄰居分布的 perplexity = exp(entropy)；sigma 越大通常涵蓋越多鄰居
squared8 = np.sum((subset8[1:] - subset8[0]) ** 2, axis=1)
for sigma8 in [5., 15., 30.]:
    logits8 = -squared8 / (2 * sigma8**2); prob8 = np.exp(logits8 - logits8.max()); prob8 /= prob8.sum()
    print('sigma/perplexity:', sigma8, np.exp(-np.sum(prob8 * np.log(prob8 + 1e-300))))

## 練習與參考答案

- 「最大變異」一定保留分類訊息嗎？**否，若標籤由小變異方向決定，PCA 可能丟掉關鍵訊息。**
- PCA 在什麼資料上操作？**中心化後的資料；`inverse_transform` 會把平均加回。**
- LDA 是無監督降維嗎？**否，它需要標籤。**
- 二維散點好看就代表模型較好嗎？**否，要用同一切分、同一分類器、把降維放進 CV 才比得準。**